# QuickPay FinTech Operations — Data Pipeline
**Student Assignment | Parts 3, 4 & 5**

This notebook contains:
- Part 3: Python Reconciliation Workflow (Ledger vs Gateway)
- Part 4: JSON Normalization (API Response)
- Part 5: Dashboard Source Files Generation

## Setup — Imports and Paths

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import os

RAW  = "../01_data/raw"
PROC = "../01_data/processed"
os.makedirs(PROC, exist_ok=True)
print("Libraries loaded. Output directory ready.")

---
## Part 3: Reconciliation Workflow
### Step 1 — Load Ledger and Gateway Files

In [ ]:
ledger  = pd.read_csv(f"{RAW}/ledger.csv")
gateway = pd.read_csv(f"{RAW}/gateway.csv")

print("=== LEDGER ===")
print(f"Shape: {ledger.shape}")
print(ledger.to_string())
print()
print("=== GATEWAY ===")
print(f"Shape: {gateway.shape}")
print(gateway.to_string())

### Step 2 — Duplicate and Null Checks

In [ ]:
print("=== DUPLICATE CHECK ===")
print(f"Ledger duplicates:  {ledger.duplicated().sum()}")
print(f"Gateway duplicates: {gateway.duplicated().sum()}")

print()
print("=== NULL CHECK — LEDGER ===")
print(ledger.isnull().sum())

print()
print("=== NULL CHECK — GATEWAY ===")
print(gateway.isnull().sum())

print()
print("Ledger Transaction IDs:",  ledger['transaction_id'].tolist())
print("Gateway Transaction IDs:", gateway['transaction_id'].tolist())

### Step 3 — Identify Records Missing in Gateway

In [ ]:
# Present in ledger but NOT in gateway
missing_in_gateway = ledger[~ledger['transaction_id'].isin(gateway['transaction_id'])].copy()
missing_in_gateway.to_csv(f"{PROC}/missing_in_gateway.csv", index=False)

print(f"Records missing in gateway: {len(missing_in_gateway)}")
print(missing_in_gateway.to_string())

### Step 4 — Identify Records Missing in Ledger

In [ ]:
# Present in gateway but NOT in ledger
missing_in_ledger = gateway[~gateway['transaction_id'].isin(ledger['transaction_id'])].copy()
missing_in_ledger.to_csv(f"{PROC}/missing_in_ledger.csv", index=False)

print(f"Records missing in ledger: {len(missing_in_ledger)}")
print(missing_in_ledger.to_string())

### Step 5 — Identify Amount Mismatches

In [ ]:
common_ids = set(ledger['transaction_id']) & set(gateway['transaction_id'])

l = ledger[ledger['transaction_id'].isin(common_ids)].set_index('transaction_id')
g = gateway[gateway['transaction_id'].isin(common_ids)].set_index('transaction_id')

amount_joined = l[['amount_usd']].join(g[['amount_usd']], lsuffix='_ledger', rsuffix='_gateway')
amount_mismatches = amount_joined[
    amount_joined['amount_usd_ledger'] != amount_joined['amount_usd_gateway']
].reset_index()
amount_mismatches['difference'] = (
    amount_mismatches['amount_usd_ledger'] - amount_mismatches['amount_usd_gateway']
).round(2)

amount_mismatches.to_csv(f"{PROC}/amount_mismatches.csv", index=False)
print(f"Amount mismatches found: {len(amount_mismatches)}")
print(amount_mismatches.to_string())

### Step 6 — Identify Status Mismatches

In [ ]:
status_joined = l[['status']].join(g[['status']], lsuffix='_ledger', rsuffix='_gateway')
status_mismatches = status_joined[
    status_joined['status_ledger'] != status_joined['status_gateway']
].reset_index()

status_mismatches.to_csv(f"{PROC}/status_mismatches.csv", index=False)
print(f"Status mismatches found: {len(status_mismatches)}")
print(status_mismatches.to_string())

### Step 7 — Build Final Reconciliation Report

In [ ]:
recon = l.join(g, lsuffix='_ledger', rsuffix='_gateway').reset_index()
recon['amount_match'] = recon['amount_usd_ledger'] == recon['amount_usd_gateway']
recon['status_match'] = recon['status_ledger'] == recon['status_gateway']
recon['has_issue'] = ~(recon['amount_match'] & recon['status_match'])
recon['issue_type'] = recon.apply(lambda r: 
    'Amount+Status Mismatch' if not r['amount_match'] and not r['status_match']
    else 'Amount Mismatch' if not r['amount_match']
    else 'Status Mismatch' if not r['status_match']
    else 'OK', axis=1)

recon.to_csv(f"{PROC}/reconciliation_report.csv", index=False)
print(f"Reconciliation report rows: {len(recon)}")
print(recon[['transaction_id','amount_usd_ledger','amount_usd_gateway',
             'status_ledger','status_gateway','amount_match','status_match','issue_type']].to_string())

### Step 8 — Generate Summary Metrics

In [ ]:
amount_at_risk = amount_mismatches['amount_usd_ledger'].sum() if len(amount_mismatches) > 0 else 0

metrics = {
    "total_ledger_rows": int(len(ledger)),
    "total_gateway_rows": int(len(gateway)),
    "missing_in_gateway_count": int(len(missing_in_gateway)),
    "missing_in_ledger_count": int(len(missing_in_ledger)),
    "amount_mismatch_count": int(len(amount_mismatches)),
    "status_mismatch_count": int(len(status_mismatches)),
    "reconciliation_issue_count": int(recon['has_issue'].sum()),
    "ledger_total_amount": round(float(ledger['amount_usd'].sum()), 2),
    "gateway_total_amount": round(float(gateway['amount_usd'].sum()), 2),
    "amount_at_risk": round(float(amount_at_risk), 2)
}

with open("../04_python/summary_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("=== SUMMARY METRICS ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")

---
## Part 4: JSON Normalization
### Step 1 — Read Nested API Response

In [ ]:
api_data = {
  "generated_at": "2026-03-07T10:00:00Z",
  "source": "QuickPay Settlement API",
  "batches": [
    {"batch_id":"B001",
     "merchant":{"merchant_id":"M001","merchant_name":"Alpha Mart","region":"APAC"},
     "settlements":[
       {"settlement_id":"S001","amount_usd":1520.5,"status":"settled",
        "processed_at":"2026-03-07T08:10:00Z","bank":{"name":"Bank A","country":"IN"}},
       {"settlement_id":"S002","amount_usd":980.0,"status":"pending",
        "processed_at":"2026-03-07T08:45:00Z","bank":{"name":"Bank A","country":"IN"}},
       {"settlement_id":"S003","amount_usd":640.0,"status":"settled",
        "processed_at":"2026-03-07T09:15:00Z","bank":{"name":"Bank B","country":"SG"}}
     ]},
    {"batch_id":"B002",
     "merchant":{"merchant_id":"M004","merchant_name":"Delta Travels","region":"US"},
     "settlements":[
       {"settlement_id":"S004","amount_usd":2100.0,"status":"settled",
        "processed_at":"2026-03-07T08:20:00Z","bank":{"name":"Bank C","country":"US"}},
       {"settlement_id":"S005","amount_usd":500.0,"status":"failed",
        "processed_at":"2026-03-07T08:50:00Z","bank":{"name":"Bank C","country":"US"}},
       {"settlement_id":"S006","amount_usd":7200.0,"status":"settled",
        "processed_at":"2026-03-07T09:30:00Z","bank":{"name":"Bank C","country":"US"}}
     ]}
  ]
}

print("Top-level keys:", list(api_data.keys()))
print("Number of batches:", len(api_data['batches']))
print("Settlements per batch:")
for b in api_data['batches']:
    print(f"  {b['batch_id']} ({b['merchant']['merchant_name']}): {len(b['settlements'])} settlements")

### Step 2 — Flatten JSON into Tabular Form

In [ ]:
rows = []
for batch in api_data['batches']:
    for s in batch['settlements']:
        rows.append({
            'batch_id'        : batch['batch_id'],
            'merchant_id'     : batch['merchant']['merchant_id'],
            'merchant_name'   : batch['merchant']['merchant_name'],
            'merchant_region' : batch['merchant']['region'],
            'settlement_id'   : s['settlement_id'],
            'amount_usd'      : s['amount_usd'],
            'status'          : s['status'],
            'processed_at'    : pd.to_datetime(s['processed_at']).strftime('%Y-%m-%d %H:%M:%S'),
            'bank_name'       : s['bank']['name'],
            'bank_country'    : s['bank']['country'],
            'api_generated_at': pd.to_datetime(api_data['generated_at']).strftime('%Y-%m-%d %H:%M:%S'),
            'api_source'      : api_data['source'],
        })

api_df = pd.DataFrame(rows)
print("Flattened shape:", api_df.shape)
print("Columns:", api_df.columns.tolist())

### Step 3 — Clean Column Names and Save

In [ ]:
# Column names already clean (lowercase, underscores)
# Verify dtypes
print("Column dtypes:")
print(api_df.dtypes)
print()
print("Normalized API data:")
print(api_df.to_string())

api_df.to_csv(f"{PROC}/api_normalized.csv", index=False)
print("\napi_normalized.csv saved.")

---
## Part 5: Dashboard Source Files

In [ ]:
ct = pd.read_csv(f"{PROC}/cleaned_transactions.csv")
print("Cleaned transactions loaded:", ct.shape)
print(ct[['transaction_id','merchant_name','status','amount_usd','gateway_region','payment_method']].head())

### Daily Summary

In [ ]:
daily = ct.groupby('transaction_date').agg(
    total_transactions   = ('transaction_id','count'),
    total_gmv_usd        = ('amount_usd','sum'),
    captured_gmv_usd     = ('amount_usd', lambda x: x[ct.loc[x.index,'status']=='captured'].sum()),
    success_count        = ('status', lambda x: (x=='captured').sum()),
    failed_count         = ('status', lambda x: (x=='failed').sum()),
    chargeback_count     = ('status', lambda x: (x=='chargeback').sum()),
).reset_index()
daily['success_rate_pct'] = (daily['success_count'] / daily['total_transactions'] * 100).round(2)
daily['total_gmv_usd']    = daily['total_gmv_usd'].round(2)
daily['captured_gmv_usd'] = daily['captured_gmv_usd'].round(2)
daily.to_csv(f"{PROC}/daily_summary.csv", index=False)
print("daily_summary.csv saved")
print(daily.to_string())

### Payment Method Breakdown

In [ ]:
pm = ct.groupby('payment_method').agg(
    transaction_count = ('transaction_id','count'),
    total_gmv_usd     = ('amount_usd','sum'),
    captured_gmv_usd  = ('amount_usd', lambda x: x[ct.loc[x.index,'status']=='captured'].sum()),
).reset_index()
pm['total_gmv_usd']    = pm['total_gmv_usd'].round(2)
pm['captured_gmv_usd'] = pm['captured_gmv_usd'].round(2)
pm.to_csv(f"{PROC}/payment_method_breakdown.csv", index=False)
print("payment_method_breakdown.csv saved")
print(pm.to_string())

### Region Breakdown

In [ ]:
rb = ct.groupby('gateway_region').agg(
    transaction_count = ('transaction_id','count'),
    total_gmv_usd     = ('amount_usd','sum'),
    captured_gmv_usd  = ('amount_usd', lambda x: x[ct.loc[x.index,'status']=='captured'].sum()),
    avg_risk_score    = ('risk_score','mean'),
    chargeback_count  = ('status', lambda x: (x=='chargeback').sum()),
    failed_count      = ('status', lambda x: (x=='failed').sum()),
).reset_index()
rb['total_gmv_usd']    = rb['total_gmv_usd'].round(2)
rb['captured_gmv_usd'] = rb['captured_gmv_usd'].round(2)
rb['avg_risk_score']   = rb['avg_risk_score'].round(2)
rb.to_csv(f"{PROC}/region_breakdown.csv", index=False)
print("region_breakdown.csv saved")
print(rb.to_string())

### Merchant Performance Summary

In [ ]:
mps = ct.groupby(['merchant_id','merchant_name','merchant_category','gateway_region']).agg(
    total_transactions = ('transaction_id','count'),
    total_gmv_usd      = ('amount_usd','sum'),
    captured_gmv_usd   = ('amount_usd', lambda x: x[ct.loc[x.index,'status']=='captured'].sum()),
    chargeback_count   = ('status', lambda x: (x=='chargeback').sum()),
    failed_count       = ('status', lambda x: (x=='failed').sum()),
    avg_risk_score     = ('risk_score','mean'),
    high_value_count   = ('high_value_flag','sum'),
    high_risk_count    = ('high_risk_flag','sum'),
).reset_index()
mps['chargeback_ratio_pct'] = (mps['chargeback_count']/mps['total_transactions']*100).round(2)
mps['total_gmv_usd']        = mps['total_gmv_usd'].round(2)
mps['captured_gmv_usd']     = mps['captured_gmv_usd'].round(2)
mps['avg_risk_score']       = mps['avg_risk_score'].round(2)
mps.to_csv(f"{PROC}/merchant_performance_summary.csv", index=False)
print("merchant_performance_summary.csv saved")
print(mps.to_string())

---
## All Outputs Verified

In [ ]:
import glob
files = glob.glob(f"{PROC}/*.csv")
print(f"Total output files: {len(files)}")
for f in sorted(files):
    df = pd.read_csv(f)
    print(f"  {os.path.basename(f):45s} → {df.shape}")

print()
print("summary_metrics.json:")
with open("../04_python/summary_metrics.json") as f:
    print(json.dumps(json.load(f), indent=2))